# 🏦 Bank Customer Segmentation & Transaction Intelligence
### End-to-End Behavioral Analytics, RFM Clustering & Machine Learning Pipeline
**Author:** M Akshay Sharma ([@akshaysharma2005](https://github.com/akshaysharma2005))

---
## 📌 Project Overview
This notebook presents an enterprise-grade financial analytics suite modeling multi-tier banking behavioral patterns.
Using SQL window functions, RFM (Recency, Frequency, Monetary) quintile scoring, and unsupervised K-Means clustering,
we uncover actionable insights regarding customer lifetime value, dormancy risks, and regional capital concentration.

In [ ]:
import os
import sqlite3
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Visual Settings
plt.style.use('dark_background')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, sans-serif'
plt.rcParams['figure.dpi'] = 120

DB_PATH = '../data/bank_analytics.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to database successfully!')

## 1. Exploratory Data Analysis & Summary Statistics

In [ ]:
# Query high-level portfolio KPIs
kpi_df = pd.read_sql_query('''
SELECT 
    (SELECT COUNT(*) FROM customers) AS total_customers,
    (SELECT COUNT(*) FROM accounts) AS total_accounts,
    (SELECT COUNT(*) FROM transactions) AS total_transactions,
    (SELECT ROUND(SUM(amount) / 1e6, 2) FROM transactions) AS total_volume_millions,
    (SELECT ROUND(SUM(balance) / 1e6, 2) FROM accounts) AS total_deposits_millions
''', conn)
kpi_df

## 2. City-Wise Financial Volume & Customer Activity

In [ ]:
city_df = pd.read_sql_query('''
SELECT 
    c.city,
    COUNT(DISTINCT c.customer_id) AS total_customers,
    COUNT(t.transaction_id) AS total_transactions,
    ROUND(SUM(t.amount) / 1e6, 2) AS volume_millions,
    ROUND(AVG(t.amount), 2) AS avg_txn_value
FROM customers c
LEFT JOIN accounts a ON c.customer_id = a.customer_id
LEFT JOIN transactions t ON a.account_id = t.account_id
GROUP BY c.city
ORDER BY volume_millions DESC
''', conn)

plt.figure(figsize=(10, 4.5))
sns.barplot(data=city_df, x='city', y='volume_millions', palette='mako')
plt.title('Regional Banking Volume Across Major Cities (₦ Millions)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Gross Volume (₦ Millions)')
plt.tight_layout()
plt.show()

## 3. RFM (Recency, Frequency, Monetary) Customer Segmentation

In [ ]:
rfm_df = pd.read_sql_query('SELECT * FROM customer_rfm_segments', conn)
print(f'Loaded {len(rfm_df)} customer profiles.')
rfm_df[['name', 'city', 'recency', 'frequency', 'monetary', 'segment']].head(10)

## 4. Unsupervised K-Means Clustering on Behavioral Vectors

In [ ]:
# Scatter plot of PCA components
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm_df, x='pca_x', y='pca_y', hue='cluster', palette='turbo', s=70, alpha=0.8
)
plt.title('K-Means Algorithmic Clusters in 2D PCA Space', fontsize=14, fontweight='bold')
plt.xlabel('Principal Component 1 (Velocity & Inflow)')
plt.ylabel('Principal Component 2 (Inactivity & Recency)')
plt.legend(title='Cluster ID', frameon=True, facecolor='#111c33')
plt.tight_layout()
plt.show()

## 5. Strategic Recommendations & Business Playbooks
1. **Champions (Top 18-25%):** Assign dedicated relationship managers and offer premium wealth advisory products.
2. **Loyal Customers:** Incentivize high-margin financial products such as fixed deposits and foreign exchange lines.
3. **Single Product Users:** Cross-sell secondary accounts (Savings + Current or Personal Loans).
4. **Dormant Accounts (>12M Inactivity):** Launch automated SMS/Email KYC refresh campaigns with zero maintenance fee incentives.